In [0]:
from pyspark.sql import functions as F

In [0]:
# Lecture Silver
df_ter = spark.read.table("transport.silver.regularite_ter")
df_tgv = spark.read.table("transport.silver.regularite_tgv")
df_transilien = spark.read.table("transport.silver.regularite_transilien")

In [0]:
# Mart 1 - TER
mart_ter = (
    df_ter
    .groupBy("region", F.date_trunc("month", "date").alias("mois"))
    .agg(
        F.avg("taux_regul").alias("taux_regul_moyen"),
        F.sum("nb_trains_annules").alias("total_trains_annules"),
        F.sum("nb_trains_programmes").alias("total_trains_programmes"),
        F.sum("nb_trains_retards").alias("total_trains_retards")
    )
)

In [0]:

mart_ter.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("transport.gold.mart_regularite_ter")

In [0]:

# Mart 2 - TGV
mart_tgv = (
    df_tgv
    .groupBy("gare_depart", "gare_arrivee", F.date_trunc("month", "date").alias("mois"))
    .agg(
        F.avg("retard_moyen_tous_trains_arrivee").alias("retard_moyen_arrivee"),
        F.avg("retard_moyen_tous_trains_depart").alias("retard_moyen_depart"),
        F.sum("nb_trains_annules").alias("total_annules"),
        F.avg("prct_retard_causes_externes").alias("prct_cause_externe"),
        F.avg("prct_retard_cause_infrastructure").alias("prct_cause_infrastructure"),
        F.avg("prct_retard_cause_mat_roulant").alias("prct_cause_materiel")
    )
)

In [0]:

mart_tgv.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("transport.gold.mart_retards_tgv")


In [0]:

# Mart 3 - Transilien
mart_transilien = (
    df_transilien
    .groupBy("ligne", "nom_ligne", F.date_trunc("month", "date").alias("mois"))
    .agg(
        F.avg("taux_regul").alias("taux_regul_moyen"),
        F.avg("ratio_trains_a_l_heure").alias("ratio_a_l_heure_moyen")
    )
)


In [0]:

mart_transilien.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("transport.gold.mart_regularite_transilien")
